In [52]:
# =======================================================
# LightGBM Regressor를 바탕으로 M2, KOSPI 값 보간
# =======================================================

In [53]:
# 1. import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib.externals.loky.backend import context
from pandas.core.reshape import encoding

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lightgbm import LGBMRegressor
from lightgbm import early_stopping, log_evaluation

In [54]:
# 2. 데이터 가져오기

global_name = "서울은행_글로발변수_월단위_202004_202604.csv"

global_df = pd.read_csv(global_name, encoding="utf-8-sig")

print(f"{global_name}의 item 갯수 {len(global_df)}")

global_df.head(2)


서울은행_글로발변수_월단위_202004_202604.csv의 item 갯수 73


,TIME,M2,KOSPI,HOUSE_SALE,RENT_SALE,FX
0,202004,2854169.7,1849.59,81.310,83.074,1225.23
1,202005,2898342.4,1965.17,81.326,83.126,1228.67


In [67]:
#3. 데이터 전처리
print('\n [결측치 확인]')
print(global_df.isnull().sum())

global_df = global_df.sort_values("TIME")

print(f"마지막줄을 제외한 데이터에 결측이 있는가 : {global_df.iloc[:-1].isna().any().any()}")
print(f"마지막줄의 m2,kospi 빼고 결측 있는가 : {global_df.iloc[-1].drop(['M2','KOSPI']).isna().any()}")


last_df = global_df.iloc[-1].copy()

train_df = global_df.iloc[1:].iloc[:-1].copy()


display(last_df)

train_df.head(2)

train_df.tail(2)



 [결측치 확인]
TIME          0
M2            1
KOSPI         1
HOUSE_SALE    0
RENT_SALE     0
FX            0
dtype: int64
마지막줄을 제외한 데이터에 결측이 있는가 : False
마지막줄의 m2,kospi 빼고 결측 있는가 : False


TIME          202604.000
M2                   NaN
KOSPI                NaN
HOUSE_SALE       102.554
RENT_SALE        101.346
FX              1487.390
Name: 72, dtype: float64

,TIME,M2,KOSPI,HOUSE_SALE,RENT_SALE,FX
70,202602,4113574.4,5575.40,100.866,100.358,1449.32
71,202603,4132066.9,5524.28,101.898,100.802,1486.64


In [19]:
# lag data
# feature engineering
# 모든 value column 이전 3개월 lag data 기반으로 이번달 예측
# 이전 t-3개월 lag 데이터 기반으로 이번 t달 에측

value_cols = [
    'M2',
    'KOSPI',
    'HOUSE_SALE',
    'RENT_SALE',
    'FX'
]

feat_cols = [

    'TIME',
    'M2',
    'KOSPI',
    'HOUSE_SALE',
    'RENT_SALE',
    'FX'

    "M2_lag1",
    "KOSPI_lag1",
    "HOUSE_SALE_lag1",
    "RENT_SALE_lag1",
    "FX_lag1"
]



# 정렬 다시 확인
global_df = global_df.sort_values("TIME")

# global_df['YEAR']     = global_df['TIME'] // 100
# global_df['MONTH']    = global_df['TIME'] %  100
# global_df['MONTH_SIN']= np.sin(2 * np.pi * global_df['MONTH'] / 12)
# global_df['MONTH_COS']= np.cos(2 * np.pi * global_df['MONTH'] / 12)

for col in value_cols:
    lag1 = f"{col}_lag1"
    # lag2 = f"{col}_lag2"
    # lag3 = f"{col}_lag3"
    # roll3_mean = f"{col}_roll3_mean"
    # roll3_std = f"{col}_roll3_std"
    # change1 = f"{col}_change1"

    global_df[lag1] = global_df[col].shift(1)
    # global_df[lag2] = global_df[col].shift(2)
    # global_df[lag3] = global_df[col].shift(3)
    #
    #
    # global_df[roll3_mean] = (global_df[col].transform(lambda x: x.shift(1).rolling(3).mean()))
    # global_df[roll3_std] = (global_df[col].transform(lambda x: x.shift(1).rolling(3).std()))
    #
    # global_df[change1] = global_df[col].shift(1).pct_change(1)



print('\n [결측치 확인]')
print(global_df.isnull().sum())

print(len(global_df))

global_df.head(2)

global_df.to_csv("feat_서울은행_글로발변수_월단위_202004_202604.csv", index=False, encoding="utf-8-sig")




 [결측치 확인]
TIME               0
M2                 1
KOSPI              1
HOUSE_SALE         0
RENT_SALE          0
FX                 0
M2_lag1            1
KOSPI_lag1         1
HOUSE_SALE_lag1    1
RENT_SALE_lag1     1
FX_lag1            1
dtype: int64
73


In [41]:
# m2 학습용 데이터 분할
# 시계열로 분할

feat_df = pd.read_csv("feat_서울은행_글로발변수_월단위_202004_202604.csv",  encoding="utf-8-sig")


cost_index = "M2"
kospi_index = "KOSPI"

train_df = feat_df[(feat_df["TIME"] >= 202005) & (feat_df["TIME"] <= 202603)]

X_train = train_df.drop(columns=cost_index)
y_train = train_df[cost_index]

X_kospi_train = train_df.drop(columns=kospi_index)
y_kospi_train = train_df[kospi_index]


display(X_train.head(2))
print("train:", X_train.shape)
print(f"train 기간: {X_train['TIME'].min()} - {X_train['TIME'].max()}")


display(X_kospi_train.head(2))
print("train:", X_kospi_train.shape)
print(f"train 기간: {X_kospi_train['TIME'].min()} - {X_kospi_train['TIME'].max()}")



,TIME,KOSPI,HOUSE_SALE,RENT_SALE,FX,M2_lag1,KOSPI_lag1,HOUSE_SALE_lag1,RENT_SALE_lag1,FX_lag1
1,202005,1965.17,81.326,83.126,1228.67,2854169.7,1849.59,81.310,83.074,1225.23
2,202006,2134.70,81.755,83.439,1210.01,2898342.4,1965.17,81.326,83.126,1228.67


train: (71, 10)
train 기간: 202005 - 202603


,TIME,M2,HOUSE_SALE,RENT_SALE,FX,M2_lag1,KOSPI_lag1,HOUSE_SALE_lag1,RENT_SALE_lag1,FX_lag1
1,202005,2898342.4,81.326,83.126,1228.67,2854169.7,1849.59,81.310,83.074,1225.23
2,202006,2919411.8,81.755,83.439,1210.01,2898342.4,1965.17,81.326,83.126,1228.67


train: (71, 10)
train 기간: 202005 - 202603


In [42]:
from sklearn.linear_model import RidgeCV, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# model = Pipeline([
#     ("scaler", StandardScaler()),
#     ("ridge", RidgeCV(
#         alphas=[0.01,0.1,1,10,100]
#     ))
# ])

m2_model = Pipeline([
    ("scale",StandardScaler()),
    ("model",ElasticNet(
        alpha=0.1,
        l1_ratio=0.5
    ))
])

m2_model.fit(X_train, y_train)


,steps,"[('scale', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,0.1
,l1_ratio,0.5
,fit_intercept,True
,precompute,False


In [43]:
kospi_model = Pipeline([
    ("scale",StandardScaler()),
    ("model",ElasticNet(
        alpha=0.1,
        l1_ratio=0.5
    ))
])

kospi_model.fit(X_kospi_train, y_kospi_train)

,steps,"[('scale', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,0.1
,l1_ratio,0.5
,fit_intercept,True
,precompute,False


In [45]:

m2_x = pd.DataFrame([{
    "TIME": 202604 ,
    "KOSPI":5524.28,
    "HOUSE_SALE":102.554,
    "RENT_SALE":101.346,
    "FX":1487.39,
    "M2_lag1":4132066.9,
    "KOSPI_lag1":5524.28,
    "HOUSE_SALE_lag1":101.898,
    "RENT_SALE_lag1":100.802,
    "FX_lag1":1486.64
}])

m2_pred = m2_model.predict(m2_x)

print(f"{m2_x['TIME']} 의 M2 예측 : {m2_pred} <- 이전달 M2 :{m2_x['M2_lag1']} ")


kospi_x = pd.DataFrame([{
    "TIME": 202604 ,
    "M2":m2_pred,
    "HOUSE_SALE":102.554,
    "RENT_SALE":101.346,
    "FX":1487.39,
    "M2_lag1":4132066.9,
    "KOSPI_lag1":5524.28,
    "HOUSE_SALE_lag1":101.898,
    "RENT_SALE_lag1":100.802,
    "FX_lag1":1486.64
}])

kospi_pred = kospi_model.predict(kospi_x)

print(f"{kospi_x['TIME']} 의 KOSPI 예측 : {kospi_pred} <- 이전달 M2 :{kospi_x['KOSPI_lag1']} ")




0    202604
Name: TIME, dtype: int64 의 M2 예측 : [4184855.57230154] <- 이전달 M2 :0    4132066.9
Name: M2_lag1, dtype: float64 
0    202604
Name: TIME, dtype: int64 의 KOSPI 예측 : [5755.37710053] <- 이전달 M2 :0    5524.28
Name: KOSPI_lag1, dtype: float64 
